# 01 · Ingestão Bronze

Leitura pura da Landing + gravação em Delta na Bronze, sem nenhuma regra de negócio.
O `ingestion_timestamp` é adicionado **na mesma cadeia que termina em `.write()`**, para reforçar
que, por causa da Lazy Evaluation do Spark, esse timestamp só é calculado quando a ação de
escrita realmente dispara — não no momento da leitura.

In [ ]:
catalog = "olist_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

In [ ]:
from pyspark.sql.functions import current_timestamp

# 1. Mapeamento dos caminhos no Volume (criado no notebook 00, schema landing)
path_customers = f"{landing_path}/olist_customers_dataset.csv"
path_orders = f"{landing_path}/olist_orders_dataset.csv"
path_products = f"{landing_path}/olist_products_dataset.csv"
path_items = f"{landing_path}/olist_order_items_dataset.csv"
path_payments = f"{landing_path}/olist_order_payments_dataset.csv"
path_category_translation = f"{landing_path}/product_category_name_translation.csv"

# 2. Leitura pura (sem transformações de negócio)
df_customers_raw = spark.read.csv(path_customers, header=True, inferSchema=True)
df_orders_raw = spark.read.csv(path_orders, header=True, inferSchema=True)
df_products_raw = spark.read.csv(path_products, header=True, inferSchema=True)
df_items_raw = spark.read.csv(path_items, header=True, inferSchema=True)
df_payments_raw = spark.read.csv(path_payments, header=True, inferSchema=True)
df_category_translation_raw = spark.read.csv(path_category_translation, header=True,)

In [ ]:
# 3. Gravação com adição do timestamp no momento da escrita
df_customers_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.customers")

df_orders_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.orders")

df_items_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.order_items")

df_products_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.products")

df_payments_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.order_payments")

df_category_translation_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.product_category_translation")

display(spark.table(f"{catalog}.bronze.orders").limit(5))